In [ ]:
import os
import pandas as pd
from pprint import pprint
from loguru import logger

def build_bioheart_dataset(input_dir: str) -> None:
    """
    Refactor and process the BioHEART dataset into pi-metaboqc standard format.
    """
    logger.info("Starting BioHEART dataset processing...")
    
    # 1. Define paths and ensure output directory exists
    meta_file = os.path.join(input_dir, "BioHEART_clinical_metadata.csv")
    int_file = os.path.join(input_dir, "BioHEART_metabolomics.csv")
    os.makedirs(output_dir, exist_ok=True)
    
    # 2. Load clinical metadata
    logger.info(f"Loading clinical metadata from {meta_file}")
    meta_df = pd.read_csv(meta_file, dtype={"Pat ID": str})
    meta_df = meta_df.set_index("Pat ID")
    
    # 3. Load and filter intensity MultiIndex DataFrame
    logger.info(f"Loading intensity matrix from {int_file}")
    int_df = pd.read_csv(int_file, header=[0, 1, 2, 3], index_col=[0])
    
    # Drop specific sample types (SR, BR, Replicate)
    type_vals = int_df.columns.get_level_values("type")
    valid_mask = ~type_vals.isin(['SR', 'BR', 'Replicate'])
    int_df = int_df.loc[:, valid_mask]
    logger.debug(f"Intensity matrix shape after filtering: {int_df.shape}")
    
    # 4. Extract column attributes and merge with clinical metadata
    # The columns have levels: ['sample', 'order', 'type', 'batch']
    col_df = int_df.columns.to_frame().reset_index(drop=True)
    
    logger.info("Merging intensity headers with clinical metadata...")
    merged_meta = col_df.join(meta_df, on="sample", how="left")
    
    # 5. Standardize column names for pi-metaboqc
    merged_meta = merged_meta.rename(columns={
        "sample": "Sample Name", 
        "order": "Inject Order",
        "type": "Sample Type", 
        "batch": "Batch"})
    
    # 6. Transform 'Sample_ID' using vectorized operations
    is_pool = merged_meta["Sample Name"] == 'Pool'
    pool_counts = is_pool.cumsum()
    
    merged_meta.loc[is_pool, "Sample Name"] = (
        'Pool-' + pool_counts[is_pool].astype(str))
    merged_meta.loc[~is_pool, "Sample Name"] = (
        'Pat-' + merged_meta.loc[~is_pool, "Sample Name"].astype(str))
    
    # 7. Standardize 'Sample_Type' and 'Batch' variables
    # Map 'S' to 'Sample' and 'Pool' to 'QC' for pipeline compatibility
    type_mapping = {"S": "Sample", "Pool": "QC"}
    merged_meta["Sample Type"] = merged_meta["Sample Type"].replace(
        type_mapping)
    
    merged_meta["Batch"] = "Batch-" + merged_meta["Batch"].astype(str).map(
        lambda x: x.zfill(2))
    merged_meta = merged_meta.fillna("Invaild")
    
    # 8. Align intensity matrix with the new Sample_IDs
    int_df.columns = merged_meta["Sample Name"]
    int_df.index.name = "Metabolite"
    
    return int_df, merged_meta


def summarize_batch_info(
    meta: pd.DataFrame,
    batch_col: str = "Batch",
    order_col: str = "Inject Order",
    type_col: str = "Sample Type"
) -> pd.DataFrame:
    """
    Summarizes batch details including injection order range, sample counts, 
    and QC counts based on the standardized metadata dataframe.
    """
    logger.info("Generating batch summary...")
    
    missing_cols = [
        c for c in [batch_col, order_col, type_col] if c not in meta.columns
    ]
    if missing_cols:
        raise ValueError(f"Missing required columns in meta: {missing_cols}")

    orders = pd.to_numeric(meta[order_col], errors="coerce")
    summary_records = []
    
    for batch_id, group in meta.groupby(batch_col):
        b_orders = orders[group.index]
        
        if b_orders.notna().any():
            order_range = f"{b_orders.min():.0f} ~ {b_orders.max():.0f}"
        else:
            order_range = "Unknown"
        
        type_counts = group[type_col].value_counts()
        
        summary_records.append({
            "Batch ID": batch_id,
            "Sample": type_counts.get("Sample", 0),
            "QC": type_counts.get("QC", 0),
            "Inject Order": order_range
        })
        
    return pd.DataFrame(summary_records)

if __name__ == "__main__":
    # Update these paths to match your local environment setup

    input_dir = os.path.abspath(
        os.path.join("..", "..", "data", "raw", "hRUV_BioHeart"))
    output_dir = os.path.abspath(
        os.path.join("..", "..", "data", "processed", "hRUV_BioHeart"))

    intensity_df, metadata_df = build_bioheart_dataset(
        input_dir=input_dir)

    batch_summary = summarize_batch_info(
        meta=metadata_df, batch_col = "Batch", order_col = "Inject Order",
        type_col = "Sample Type")
    print("\n===== Batch Information Summary =====")
    print(batch_summary.to_markdown(index=False,stralign="center", numalign="center"))
    print("=====================================\n")

    # 3. Export clean data
    os.makedirs(output_dir, exist_ok=True)
    meta_path = os.path.join(output_dir, "project_meta.csv")
    int_path = os.path.join(output_dir, "project_intensity.csv")

    logger.info("Saving standardized datasets...")
    metadata_df.to_csv(meta_path, index=False, encoding="utf-8-sig")
    intensity_df.to_csv(int_path, encoding="utf-8-sig", na_rep="N/A")


2026-05-26 08:35:24.146 | INFO     | __main__:build_bioheart_dataset:10 - Starting BioHEART dataset processing...
2026-05-26 08:35:24.147 | INFO     | __main__:build_bioheart_dataset:18 - Loading clinical metadata from c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\raw\hRUV_BioHeart\BioHEART_clinical_metadata.csv
2026-05-26 08:35:24.150 | INFO     | __main__:build_bioheart_dataset:23 - Loading intensity matrix from c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\raw\hRUV_BioHeart\BioHEART_metabolomics.csv
2026-05-26 08:35:24.197 | DEBUG    | __main__:build_bioheart_dataset:30 - Intensity matrix shape after filtering: (100, 1166)
2026-05-26 08:35:24.199 | INFO     | __main__:build_bioheart_dataset:36 - Merging intensity headers with clinical metadata...
2026-05-26 08:35:24.206 | INFO     | __main__:summarize_batch_info:82 - Generating batch summary...
2026-05-26 08:35:24.218 | INFO     | __main__:<module>:135 - Saving standardized dat


===== Batch Information Summary =====
|  Batch ID  |  Sample  |  QC  |  Injection Order  |
|:----------:|:--------:|:----:|:-----------------:|
|  Batch-01  |    73    |  10  |      1 ~ 89       |
|  Batch-02  |    66    |  11  |     90 ~ 180      |
|  Batch-03  |    62    |  11  |     181 ~ 267     |
|  Batch-04  |    70    |  11  |     268 ~ 362     |
|  Batch-05  |    66    |  10  |     363 ~ 452     |
|  Batch-06  |    67    |  11  |     453 ~ 542     |
|  Batch-07  |    66    |  11  |     543 ~ 632     |
|  Batch-08  |    66    |  10  |     633 ~ 721     |
|  Batch-09  |    66    |  11  |     722 ~ 812     |
|  Batch-10  |    66    |  11  |     813 ~ 903     |
|  Batch-11  |    67    |  11  |     904 ~ 994     |
|  Batch-12  |    66    |  11  |    995 ~ 1085     |
|  Batch-13  |    66    |  11  |    1086 ~ 1176    |
|  Batch-14  |    66    |  11  |    1177 ~ 1267    |
|  Batch-15  |    71    |  11  |    1268 ~ 1361    |



In [30]:
pprint(intensity_df.shape)
pprint(intensity_df.head())

(100, 1166)
Sample Name                       Pool-1        Pool-2         Pat-1  \
Metabolite                                                             
1-methylhistamine           1.045950e+07  1.351577e+07  1.253405e+07   
2-Arachidonyl glycerol      6.855405e+04  8.499870e+04  7.688991e+04   
2-Methylbutyrylcarnitine.1           NaN           NaN           NaN   
2-Methylbutyrylcarnitine.2           NaN           NaN           NaN   
2-Methylbutyrylcarnitine.3           NaN           NaN           NaN   

Sample Name                        Pat-3        Pat-21        Pat-22  \
Metabolite                                                             
1-methylhistamine           1.473340e+07  1.786469e+07  1.616507e+07   
2-Arachidonyl glycerol      9.484460e+04  7.702922e+04  7.624380e+04   
2-Methylbutyrylcarnitine.1           NaN           NaN           NaN   
2-Methylbutyrylcarnitine.2           NaN           NaN           NaN   
2-Methylbutyrylcarnitine.3           NaN           

In [31]:
pprint(metadata_df.head())
pprint(metadata_df["Sample Type"].value_counts())

  Sample Name Inject Order Sample Type     Batch      Age   Gender      HTN
0      Pool-1            1          QC  Batch-01  Invaild  Invaild  Invaild
1      Pool-2            2          QC  Batch-01  Invaild  Invaild  Invaild
2       Pat-1            3      Sample  Batch-01     59.0        M      0.0
3       Pat-3            4      Sample  Batch-01     66.0        M      0.0
4      Pat-21            5      Sample  Batch-01     35.0        M      0.0
Sample Type
Sample    1004
QC         162
Name: count, dtype: int64
